In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.transforms import transforms as T
from torch.utils.data import DataLoader,Subset

In [5]:
t = T.ToTensor()
trd = DataLoader(Subset(torchvision.datasets.CIFAR10('./data',train=True, download=True, transform=t), range(200)), batch_size=10, shuffle=True)
ted = DataLoader(Subset(torchvision.datasets.CIFAR10('./data', False, download=True, transform=t),range(50)),batch_size=10,shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


In [6]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(4096,512), nn.ReLU(), nn.Dropout(.5), nn.Linear(512,10)
        )
    
    def forward(self,x):
        return self.net(x)

In [9]:
def train(model,opt,crit,epochs):
    for e in range(epochs):
        model.train()
        rl=0
        tc=0
        tn=0
        for x,y in trd:
            opt.zero_grad()
            o=model(x)
            l=crit(o,y) 
            l.backward()
            opt.step()
            rl += l.item()
            tc += (o.argmax(1)==y).sum().item()
            tn += len(y)

        model.eval()
        tl=0
        c=0
        n=0
        with torch.no_grad():
            for x,y in ted:
                o=model(x)
                l=crit(o,y)
                tl += l.item()
                c += (o.argmax(1)==y).sum().item()
                n += len(y)
        print(f'Epoch : {e+1} | Train Loss: {rl/len(trd):.2f} Accuracy: {tc/tn*100:.2f}% | Test Loss: {tl/len(ted):.2f} Accuracy: {c/n*100:.2f}%')

model = CNN()
opt = optim.Adam(model.parameters(),1e-3)
crit = nn.CrossEntropyLoss()
train(model,opt,crit,30)

Epoch : 1 | Train Loss: 3.79 Accuracy: 10.50% | Test Loss: 2.36 Accuracy: 12.00%
Epoch : 2 | Train Loss: 2.37 Accuracy: 17.00% | Test Loss: 2.12 Accuracy: 16.00%
Epoch : 3 | Train Loss: 2.01 Accuracy: 28.50% | Test Loss: 1.98 Accuracy: 24.00%
Epoch : 4 | Train Loss: 1.66 Accuracy: 39.00% | Test Loss: 1.85 Accuracy: 26.00%
Epoch : 5 | Train Loss: 1.48 Accuracy: 50.00% | Test Loss: 2.02 Accuracy: 28.00%
Epoch : 6 | Train Loss: 1.31 Accuracy: 57.50% | Test Loss: 1.95 Accuracy: 22.00%
Epoch : 7 | Train Loss: 1.31 Accuracy: 56.00% | Test Loss: 2.03 Accuracy: 22.00%
Epoch : 8 | Train Loss: 1.06 Accuracy: 64.50% | Test Loss: 2.15 Accuracy: 30.00%
Epoch : 9 | Train Loss: 0.89 Accuracy: 67.00% | Test Loss: 1.98 Accuracy: 34.00%
Epoch : 10 | Train Loss: 0.88 Accuracy: 69.50% | Test Loss: 2.14 Accuracy: 26.00%
Epoch : 11 | Train Loss: 0.78 Accuracy: 72.00% | Test Loss: 2.22 Accuracy: 24.00%
Epoch : 12 | Train Loss: 0.68 Accuracy: 78.50% | Test Loss: 2.41 Accuracy: 28.00%
Epoch : 13 | Train Loss: 